# 15 Debate Agents with Consensus Voting using LangGraph

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Modela **agentes en debate con votación de consenso**. Tres agentes (`agent_a`,
`agent_b`, `agent_c`), cada uno con una postura (*stance*) distinta, argumentan de forma
secuencial sobre un tema; al final, un nodo `judge` resume el debate y determina el
consenso/veredicto.

El notebook combina, en orden de dependencia, `llm_provider` (`ChatAnthropic`),
`debate_graph` (grafo de debate) y `main` (lee el tema e imprime la traza del debate).

## Ejemplo de uso

**Datos de interacción que espera el agente.** Los tres agentes debaten en secuencia y el
`judge` cierra con el veredicto; sin pausa, concluye en un `invoke`.

- Entrada inicial esperada: `{"messages": [HumanMessage(content="<tema de debate>")]}`.
- La salida contiene los argumentos de `agent_a/b/c` y el resumen del `judge`.

```python
from langchain_core.messages import HumanMessage

llm = get_llm()
app = build_graph(llm)
final_state = app.invoke(
    {"messages": [HumanMessage(content="¿Conviene usar microservicios en startups?")]}
)                                                   # agent_a→agent_b→agent_c→judge
for i, m in enumerate(final_state["messages"], 1):
    name = getattr(m, "name", None) or "user/system"
    print(f"{i:02d}. [{name.upper()}] {m.content}\n")
```

In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

In [2]:

# Load environment variables from .env file
load_dotenv()

True

In [3]:


def get_llm():
    # Read API key and model name from environment
    api_key = os.getenv("ANTHROPIC_API_KEY")
    model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

    # Fail fast if API key is missing
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY is missing in .env")

    # Initialize and return Claude chat model
    return ChatAnthropic(
        model=model,
        api_key=api_key,
        temperature=0,  # Deterministic output
    )

In [4]:
from typing import List, TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END

In [5]:


class State(MessagesState):
    """Shared state for debate and voting."""
    votes: List[str]  # Stores YES/NO votes from agents

In [6]:


def debate_agent(llm, name: str, stance: str):
    def node(state: State) -> dict:
        # Agent presents its argument
        print(f"→ {name.upper()} presenting argument...")

        # Debate topic from last user message
        topic = state["messages"][-1].content

        # Role and stance definition for the agent
        system_prompt = SystemMessage(
            content=(
                f"Role: Debate Agent\n"
                f"Stance: {stance}\n\n"
                "Provide a concise argument.\n"
                "End clearly with either YES or NO."
            )
        )

        response = llm.invoke([system_prompt, HumanMessage(content=topic)])
        content = response.content.strip()

        # Extract vote from agent response
        vote = "YES" if "YES" in content.upper() else "NO"

        # Append vote and message to shared state
        return {
            "votes": [vote],
            "messages": [HumanMessage(content=content, name=name)]
        }

    return node

In [7]:


def judge_node():
    def node(state: State) -> dict:
        # Aggregate votes from all agents
        print("→ JUDGE aggregating votes...")

        votes = state.get("votes", [])
        yes_votes = votes.count("YES")
        no_votes = votes.count("NO")

        # Majority decision logic
        if yes_votes > no_votes:
            decision = "YES"
        elif no_votes > yes_votes:
            decision = "NO"
        else:
            decision = "TIE"

        summary = (
            f"Votes collected: {votes}\n"
            f"Final decision (majority voting): {decision}"
        )

        # Append judge decision to message history
        return {
            "messages": [HumanMessage(content=summary, name="judge")]
        }

    return node

In [8]:


def build_graph(llm):
    graph = StateGraph(State)

    # Add debate agents
    graph.add_node(
        "agent_a",
        debate_agent(llm, "agent_a", "Strongly support the proposal"),
    )
    graph.add_node(
        "agent_b",
        debate_agent(llm, "agent_b", "Oppose the proposal"),
    )
    graph.add_node(
        "agent_c",
        debate_agent(llm, "agent_c", "Neutral analytical perspective"),
    )

    # Add judge node
    graph.add_node("judge", judge_node())

    # Define debate flow
    graph.add_edge(START, "agent_a")
    graph.add_edge("agent_a", "agent_b")
    graph.add_edge("agent_b", "agent_c")
    graph.add_edge("agent_c", "judge")
    graph.add_edge("judge", END)

    return graph.compile()

In [10]:
from langchain_core.messages import HumanMessage


In [11]:

print(" Debate Agents with Consensus Voting (LangGraph + Claude)")

 Debate Agents with Consensus Voting (LangGraph + Claude)


In [12]:

def print_trace(messages):
    # Display ordered debate messages
    print("\n--- Debate Trace ---\n")
    for i, m in enumerate(messages, start=1):
        name = getattr(m, "name", None)
        role = name.upper() if name else "USER/SYSTEM"

        print(f"{i:02d}. [{role}]")
        print(m.content)
        print()

In [13]:


def main():
   
 # Initialize LLM and debate graph
    llm = get_llm()
    app = build_graph(llm)

    # Read debate topic
    topic = input("Enter a debate topic (or exit()): ").strip()
    if topic.lower() in {"exit()", "exit", "quit"}:
        print("\nExiting.\n")
        return

    # Validate input
    if not topic:
        print("Please enter a valid topic.\n")
        return

    print("\n--- Debate Started ---\n")

    # Invoke debate graph
    final_state = app.invoke(
        {"messages": [HumanMessage(content=topic)]}
    )

    # Print debate results
    print_trace(final_state["messages"])
    print("-" * 70 + "\n")

In [14]:


if __name__ == "__main__":
    main()


--- Debate Started ---

→ AGENT_A presenting argument...
→ AGENT_B presenting argument...
→ AGENT_C presenting argument...
→ JUDGE aggregating votes...

--- Debate Trace ---

01. [USER/SYSTEM]
Por que el cielo es azul?

02. [AGENT_A]
## Argumento a favor de que el cielo es azul

El cielo es azul debido al **fenómeno de dispersión de Rayleigh**. Cuando la luz solar (que contiene todos los colores del espectro visible) entra en la atmósfera terrestre, choca con las moléculas de gas (principalmente nitrógeno y oxígeno).

La dispersión de Rayleigh establece que **la luz de menor longitud de onda se dispersa con mucho mayor intensidad** — específicamente, la dispersión es inversamente proporcional a la cuarta potencia de la longitud de onda (λ⁻⁴).

La luz azul tiene una longitud de onda corta (~450 nm), por lo que se dispersa aproximadamente **5-10 veces más** que la luz roja (~700 nm). Esto significa que la luz azul se esparce en todas direcciones a través del cielo, haciendo que cuando m